# Tutorial 02: Forward Kinematics

<p align="center">
  <img src="RRRP_SCARA.png" alt="RRRP SCARA Robot" width="500"/>
</p>

<p align="center"><i>
Source: <b>MODERN ROBOTICS: MECHANICS, PLANNING, AND CONTROL</b><br/>
Figure 4.12: An RRRP SCARA robot for performing pick-and-place operations.
</i></p>

---

**Prerequisite**: [Tutorial 01: Building Your First Robot](01_build_robot.ipynb)

In this tutorial, we will explore **Forward Kinematics (FK)** for an RRRP SCARA robot using two approaches:

1. **Direct Geometric Method**: Computing FK using explicit geometric formulas
2. **Denavit-Hartenberg (DH) Method**: Computing FK using systematic DH parameter conventions and homogeneous transformation matrices

## Learning Objectives

By the end of this tutorial, you will be able to:

- Derive and implement forward kinematics using geometric reasoning
- Apply the DH parameter convention systematically
- Construct homogeneous transformation matrices
- Verify that both methods produce identical results
- Visualize robot motion and compare FK predictions with simulation

## The RRRP SCARA Robot

The robot we're using is the **SCARA** built in Tutorial 01, loaded from `scara_rrrp.xml`:

| Joint | Type | Description |
|-------|------|-------------|
| θ₁ | Revolute (R) | Shoulder rotation around vertical axis |
| θ₂ | Revolute (R) | Elbow rotation around vertical axis |
| θ₃ | Revolute (R) | Wrist rotation around vertical axis |
| d₄ | Prismatic (P) | Vertical extension of the end-effector |


## Setup and Imports

First, we import the necessary libraries:
- **newton**: The Newton physics engine for robot simulation
- **warp (wp)**: NVIDIA's high-performance computing framework used by Newton
- **numpy**: For numerical computations and array operations
- **tqdm**: For progress bars during animations

In [1]:
import newton
import warp as wp
import numpy as np
from tqdm.notebook import trange
import os

# Initialize Warp
wp.init()

# Set NumPy print options for cleaner output
np.set_printoptions(precision=6, suppress=True, linewidth=100)


Warp 1.11.0.dev20251123 initialized:
   Git commit: 8b8f0b85ca54c0026574f834764e26615056aef6
   CUDA Toolkit 12.8, Driver 13.0
   Devices:
     "cpu"      : "x86_64"
     "cuda:0"   : "NVIDIA L40S" (44 GiB, sm_89, mempool enabled)
   Kernel cache:
     /root/.cache/warp/1.11.0.dev20251123


## Robot Parameters

The key dimensions of our SCARA robot are:
- **L₀ (base height)**: 0.10 m
- **L₁ (first arm length)**: 0.30 m  
- **L₂ (second arm length)**: 0.20 m
- **Prismatic stroke**: 0.0 to 0.25 m


In [ ]:
# Robot dimensions (meters)
L0 = 0.10            # Base height (height to first joint)
L1 = 0.30            # Length of first horizontal arm
L2 = 0.20            # Length of second horizontal arm

# Joint limits (from the MJCF model)
JOINT_LIMITS = np.array([
    [-np.pi/2,  np.pi/2],      # θ₁: [-90°, 90°]
    [-np.pi/4,  np.pi/4],      # θ₂: [-45°, 45°]
    [-2*np.pi/3, np.pi],       # θ₃: [-120°, 180°]
    [0.0,       0.25]          # θ₄: [0, 0.25m] (prismatic)
])

print("Robot Parameters:")
print(f"  L₀ (base height): {L0} m")
print(f"  L₁ (arm 1 length): {L1} m")
print(f"  L₂ (arm 2 length): {L2} m")
print(f"\nJoint limits:")
print(f"  θ₁: [{np.degrees(JOINT_LIMITS[0,0]):.1f}°, {np.degrees(JOINT_LIMITS[0,1]):.1f}°]")
print(f"  θ₂: [{np.degrees(JOINT_LIMITS[1,0]):.1f}°, {np.degrees(JOINT_LIMITS[1,1]):.1f}°]")
print(f"  θ₃: [{np.degrees(JOINT_LIMITS[2,0]):.1f}°, {np.degrees(JOINT_LIMITS[2,1]):.1f}°]")
print(f"  θ₄: [{JOINT_LIMITS[3,0]:.2f}m, {JOINT_LIMITS[3,1]:.2f}m]")


Robot Parameters:
  L₀ (base height): 0.1 m
  L₁ (arm 1 length): 0.3 m
  L₂ (arm 2 length): 0.2 m

Joint limits:
  θ₁: [-90.0°, 90.0°]
  θ₂: [-45.0°, 45.0°]
  θ₃: [-120.0°, 180.0°]
  θ₄: [0.00m, 0.25m]


## Part 1: Direct Geometric Forward Kinematics

### Geometric Analysis

For a SCARA robot, the forward kinematics can be derived by simple geometric reasoning:

**Horizontal Position (X-Y plane):**

The end-effector's horizontal position is the sum of the projections of both arms:

$$x = \ell_1 \cos(\theta_1) + \ell_2 \cos(\theta_1 + \theta_2)$$

$$y = \ell_1 \sin(\theta_1) + \ell_2 \sin(\theta_1 + \theta_2)$$

**Vertical Position (Z-axis):**

The height is determined by the base height and prismatic extension:

$$z = \ell_0 + d_4$$

Note that θ₃ (wrist rotation) does not affect the position of the end-effector - it only changes orientation.

### Implementation

Below we implement the `fk_scara_direct()` function that computes the end-effector position using these geometric formulas. We then test it with the home position where all angles are zero and the prismatic joint is extended by 0.1m.


In [ ]:
def fk_scara_direct(q):
    """
    Forward kinematics using direct geometric formulas.
    
    The SCARA robot has joints that all rotate about the Z-axis,
    making the geometric FK particularly simple.
    
    Parameters:
        q: array [θ₁, θ₂, θ₃, θ₄] - joint angles (rad) and prismatic displacement (m)
    
    Returns:
        p: array [x, y, z] - end-effector position in world frame (m)
    """
    theta1, theta2, theta3, theta4 = q
    
    # Horizontal position: sum of arm projections
    x = L1 * np.cos(theta1) + L2 * np.cos(theta1 + theta2)
    y = L1 * np.sin(theta1) + L2 * np.sin(theta1 + theta2)
    
    # Vertical position: base height + prismatic extension
    z = L0 + theta4
    
    return np.array([x, y, z])


# Test with home position (all zeros except θ₄=0.1)
q_home = np.array([0.0, 0.0, 0.0, 0.1])
pos_home = fk_scara_direct(q_home)

print("Home Position Test:")
print(f"  Joint values: θ₁={q_home[0]:.3f}, θ₂={q_home[1]:.3f}, θ₃={q_home[2]:.3f}, θ₄={q_home[3]:.3f}")
print(f"  EE Position: x={pos_home[0]:.4f}, y={pos_home[1]:.4f}, z={pos_home[2]:.4f}")
print(f"\n  Expected: x={L1+L2:.4f}, y=0.0000, z={L0+0.1:.4f}")


Home Position Test:
  Joint values: θ₁=0.000, θ₂=0.000, θ₃=0.000, θ₄=0.100
  EE Position: x=0.5000, y=0.0000, z=0.2000


NameError: name 'TABLE_HEIGHT' is not defined

## Part 2: Denavit-Hartenberg Forward Kinematics

### The DH Convention

The Denavit-Hartenberg convention provides a systematic way to describe the kinematic chain of a robot using four parameters per joint:

| Parameter | Symbol | Description |
|-----------|--------|-------------|
| Link length | a | Distance along $x_i$ from $z_{i-1}$ to $z_i$ |
| Link twist | α | Angle from $z_{i-1}$ to $z_i$ about $x_i$ |
| Link offset | d | Distance along $z_{i-1}$ from $x_{i-1}$ to $x_i$ |
| Joint angle | θ | Angle from $x_{i-1}$ to $x_i$ about $z_{i-1}$ |

### DH Transformation Matrix

Each joint contributes a 4×4 homogeneous transformation matrix:

$$A_i = \begin{bmatrix} 
\cos\theta_i & -\sin\theta_i & 0 & a_i \\
\sin\theta_i \cos\alpha_i & \cos\theta_i \cos\alpha_i & -\sin\alpha_i & -d_i \sin\alpha_i \\
\sin\theta_i \sin\alpha_i & \cos\theta_i \sin\alpha_i & \cos\alpha_i & d_i \cos\alpha_i \\
0 & 0 & 0 & 1
\end{bmatrix}$$

The total transformation from base to end-effector is:

$$T^{0}_n = A_1 \cdot A_2 \cdot A_3 \cdot ... \cdot A_n$$

### Implementation of the DH Transformation Matrix

Below we implement the `A_dh()` function that computes a single DH transformation matrix given the four parameters (a, α, d, θ). This function will be used to build the complete kinematic chain.


In [ ]:
def A_dh(a, alpha, d, theta):
    """
    Compute the standard Denavit-Hartenberg homogeneous transformation matrix.
    
    Parameters:
        a: link length (m)
        alpha: link twist (rad)
        d: link offset (m)
        theta: joint angle (rad)
    
    Returns:
        A: 4×4 homogeneous transformation matrix
    """
    ct = np.cos(theta)
    st = np.sin(theta)
    ca = np.cos(alpha)
    sa = np.sin(alpha)
    
    A = np.array([
        [ct,    -st,     0,    a    ],
        [st*ca,  ct*ca, -sa,  -d*sa ],
        [st*sa,  ct*sa,  ca,   d*ca ],
        [0,      0,      0,    1    ]
    ])
    
    return A


print("DH Transformation Matrix Function Created")
print("\nExample - Identity transformation (a=0, α=0, d=0, θ=0):")
print(A_dh(0, 0, 0, 0))


### DH Parameters for RRRP SCARA

The DH parameters for our SCARA robot:

| Joint i | aᵢ | αᵢ | dᵢ | θᵢ |
|---------|-----|----|----|--------|
| 1 (Shoulder) | 0 | 0 | L₀ | θ₁ (variable) |
| 2 (Elbow) | L₁ | 0 | 0 | θ₂ (variable) |
| 3 (Wrist) | L₂ | 0 | 0 | θ₃ (variable) |
| 4 (Prismatic) | 0 | 0 | d₄ (variable) | 0 |

**Key observations:**
- All α values are 0 because all joint axes are parallel (all Z-axes)
- Joint 4 has θ=0 and variable d (prismatic joint)
- The base height is captured in d₁

### Implementation of DH-based Forward Kinematics

Now we implement `fk_scara_dh()` that chains four DH transformation matrices together to compute the end-effector position. Each matrix represents one joint's contribution to the overall transformation.


In [ ]:
def fk_scara_dh(q):
    """
    Forward kinematics using Denavit-Hartenberg parameters.
    
    DH Parameter Table for RRRP SCARA:
    ┌───────┬─────┬─────┬──────────────────┬─────────────┐
    │ Joint │  a  │  α  │        d         │      θ      │
    ├───────┼─────┼─────┼──────────────────┼─────────────┤
    │   1   │  0  │  0  │       L₀         │  θ₁ (var)   │
    │   2   │ L₁  │  0  │       0          │  θ₂ (var)   │
    │   3   │ L₂  │  0  │       0          │  θ₃ (var)   │
    │   4   │  0  │  0  │     d₄ (var)     │      0      │
    └───────┴─────┴─────┴──────────────────┴─────────────┘
    
    Parameters:
        q: array [θ₁, θ₂, θ₃, θ₄] - joint variables
    
    Returns:
        p: array [x, y, z] - end-effector position in world frame
    """
    theta1, theta2, theta3, theta4 = q
    
    # Build individual transformation matrices
    T01 = A_dh(a=0,  alpha=0, d=L0, theta=theta1)
    T12 = A_dh(a=L1, alpha=0, d=0, theta=theta2)
    T23 = A_dh(a=L2, alpha=0, d=0, theta=theta3)
    T34 = A_dh(a=0,  alpha=0, d=theta4, theta=0)
    
    # Chain all transformations
    T = T01 @ T12 @ T23 @ T34
    
    # Extract position (last column, first three rows)
    return T[:3, 3]


# Test with home position
pos_dh = fk_scara_dh(q_home)
print("DH Method Test:")
print(f"  Joint values: θ₁={q_home[0]:.3f}, θ₂={q_home[1]:.3f}, θ₃={q_home[2]:.3f}, θ₄={q_home[3]:.3f}")
print(f"  EE Position: x={pos_dh[0]:.4f}, y={pos_dh[1]:.4f}, z={pos_dh[2]:.4f}")


## Part 3: Verification - Comparing Both Methods

A crucial step in developing forward kinematics is to verify that different methods produce consistent results. Let's compare the direct geometric method with the DH method across many random configurations.

### Consistency Check Function

The function below generates random joint configurations within the allowed limits, computes the end-effector position using both methods, and measures the difference. If our implementations are correct, both methods should produce identical results (within floating-point precision).


In [ ]:
def check_fk_consistency(num_tests=10, verbose=True):
    """
    Compare fk_scara_direct and fk_scara_dh on random configurations.
    
    Parameters:
        num_tests: number of random configurations to test
        verbose: whether to print detailed results
    
    Returns:
        max_error: maximum position error across all tests
    """
    print("=" * 60)
    print("FK Consistency Check: Direct vs DH Methods")
    print("=" * 60)
    
    max_error = 0.0
    all_errors = []
    
    for k in range(num_tests):
        # Generate random joint configuration within limits
        q_rand = np.random.uniform(JOINT_LIMITS[:, 0], JOINT_LIMITS[:, 1])
        
        # Compute FK using both methods
        p_direct = fk_scara_direct(q_rand)
        p_dh = fk_scara_dh(q_rand)
        
        # Calculate error (Euclidean distance)
        error = np.linalg.norm(p_direct - p_dh)
        max_error = max(max_error, error)
        all_errors.append(error)
        
        if verbose:
            print(f"\nTest {k+1}:")
            print(f"  q = [{q_rand[0]:7.4f}, {q_rand[1]:7.4f}, {q_rand[2]:7.4f}, {q_rand[3]:7.4f}]")
            print(f"  Direct: [{p_direct[0]:.6f}, {p_direct[1]:.6f}, {p_direct[2]:.6f}]")
            print(f"  DH:     [{p_dh[0]:.6f}, {p_dh[1]:.6f}, {p_dh[2]:.6f}]")
            print(f"  Error:  {error:.2e}")
    
    print("\n" + "=" * 60)
    print(f"Results over {num_tests} random configurations:")
    print(f"  Maximum error: {max_error:.2e} m")
    print(f"  Mean error:    {np.mean(all_errors):.2e} m")
    
    if max_error < 1e-10:
        print("\n✓ EXCELLENT: Both FK methods are numerically IDENTICAL")
    elif max_error < 1e-6:
        print("\n✓ PASS: Both FK methods are CONSISTENT")
    else:
        print("\n⚠ WARNING: Methods differ significantly!")
    print("=" * 60)
    
    return max_error


# Run the consistency check
_ = check_fk_consistency(num_tests=10, verbose=True)


## Part 4: Loading the SCARA Robot in Newton

Now let's load our SCARA robot from the MJCF (MuJoCo XML) file and visualize it using Newton's `ViewerRerun`.

### Loading the MJCF Model

We use Newton's `ModelBuilder` to load the robot definition from the `scara_rrrp.xml` file. This file contains the complete robot description including geometry, joints, and physical properties.


In [ ]:
# Load the SCARA robot from MJCF
builder = newton.ModelBuilder()

# Get the path to the MJCF file (in the same directory)
mjcf_path = os.path.join(os.getcwd(), "scara_rrrp.xml")

# Import the MJCF file (including table)
builder.add_mjcf(mjcf_path)

print(f"MJCF file loaded: {mjcf_path}")
print(f"Model contains: {builder.body_count} bodies, {builder.joint_count} joints")


### Finalizing the Model and Setting Initial Configuration

After loading the MJCF, we finalize the model to create the simulation-ready structure. We then set the initial joint configuration (θ₁=0, θ₂=0, θ₃=0, θ₄=0.1m) and call `newton.eval_fk()` to compute the body positions from the joint angles.


In [ ]:
# Finalize the model
model = builder.finalize()
state = model.state()

# =============================================================================
# Site Position Lookup (equivalent to MuJoCo's sensor)
# -----------------------------------------------------------------------------
# In MuJoCo, we use a sensor attached to ee_site to get the fingertip position:
#   ee_sensor_id = mj.mj_name2id(model, mj.mjtObj.mjOBJ_SENSOR, "ee_pos_sensor")
#   p_sens = data.sensordata[ee_sensor_adr : ee_sensor_adr + ee_sensor_dim]
#
# In Newton, we find the site shape and compute its world position from
# the body transform and the site's local transform.
# =============================================================================

# Find the ee_site shape index (similar to MuJoCo's sensor lookup)
ee_site_idx = None
for i, key in enumerate(model.shape_key):
    if "ee_site" in key:
        ee_site_idx = i
        break

if ee_site_idx is not None:
    print(f"Found ee_site at shape index: {ee_site_idx}")
else:
    print("Warning: ee_site not found!")

# Build a mapping of shape names to indices and colors from MJCF
# Colors extracted directly from scara_rrrp.xml geom rgba attributes
# Format: pattern -> (R, G, B) - pattern matches against Newton's shape_key

SHAPE_COLORS = {
    # Geoms with explicit names in MJCF
    "L1_start_joint": (0.7, 0.4, 0.1),   # line 93: rgba="0.7 0.4 0.1 1.0"
    "L1_end_joint": (0.7, 0.4, 0.1),     # line 94: rgba="0.7 0.4 0.1 1.0"
    "L2_start_joint": (0.0, 0.8, 0.8),   # line 106: rgba="0.0 0.8 0.8 1.0"
    "L2_end_joint": (0.0, 0.8, 0.8),     # line 107: rgba="0.0 0.8 0.8 1.0"
    "palm": (0.95, 0.8, 0.1),            # line 130: rgba="0.95 0.8 0.1 1.0"
    "left_finger": (0.8, 0.0, 0.8),      # line 133: rgba="0.8 0.0 0.8 1.0"
    "right_finger": (0.4, 0.6, 0.9),     # line 136: rgba="0.4 0.6 0.9 1.0"
    
    # Geoms WITHOUT names - Newton auto-names them based on body name
    # These need to match the body name pattern
    "table": (0.7, 0.7, 0.7),            # line 69: rgba="0.7 0.7 0.7 1.0"
    "base": (0.3, 0.3, 0.3),             # line 76: rgba="0.3 0.3 0.3 1.0"
    "piston": (0.6, 0.2, 0.8),           # line 124: rgba="0.6 0.2 0.8 1.0"
    "sleeve": (0.5, 0.5, 0.8),           # line 115: rgba="0.5 0.5 0.8 1.0"
    "ee": (0.95, 0.8, 0.1),              # ee body contains palm - fallback
    "L0": (0.2, 0.5, 0.9),               # line 82: rgba="0.2 0.5 0.9 1.0"
    "L1": (0.7, 0.4, 0.1),               # line 90: rgba="0.7 0.4 0.1 1.0"
    "L2": (0.0, 0.8, 0.8),               # line 103: rgba="0.0 0.8 0.8 1.0"
}

# Sort by name length (longest first) to match specific names before general ones
sorted_colors = sorted(SHAPE_COLORS.items(), key=lambda x: len(x[0]), reverse=True)

# Find shape indices by name - print ALL shape keys for debugging
shape_color_map = {}
print("=" * 60)
print("Newton shape keys and color matching:")
print("=" * 60)
for i, key in enumerate(model.shape_key):
    matched = False
    for name, color in sorted_colors:
        if name in key:
            shape_color_map[i] = color
            print(f"[{i:2d}] '{key}' -> '{name}' = {color}")
            matched = True
            break
    if not matched:
        print(f"[{i:2d}] '{key}' -> NO MATCH")
        
print("=" * 60)
print(f"Total shapes: {len(model.shape_key)}, Matched: {len(shape_color_map)}")
print("=" * 60)


def get_site_transform(model, state, site_idx):
    """
    Get site world position and rotation from Newton state.
    Similar to MuJoCo's data.sensordata for a framepos sensor attached to a site.
    
    This computes: world_pos = body_transform * site_local_transform
    
    Returns:
        tuple: (position, rotation_matrix) where position is (3,) and rotation is (3,3)
    """
    from scipy.spatial.transform import Rotation
    
    # Get site's parent body
    body_idx = model.shape_body.numpy()[site_idx]
    
    # Get site's local transform
    site_local = model.shape_transform.numpy()[site_idx]
    site_pos_local = site_local[:3]
    site_quat_local = site_local[3:7]
    
    if body_idx >= 0:
        # Get body world transform
        body_q = state.body_q.numpy()
        if body_q.ndim > 1:
            body_transform = body_q[body_idx]
        else:
            body_transform = body_q
        
        body_pos = body_transform[:3]
        body_quat = body_transform[3:7]  # (x, y, z, w) format
        
        # Body rotation
        r_body = Rotation.from_quat([body_quat[0], body_quat[1], body_quat[2], body_quat[3]])
        
        # Site local rotation
        r_site_local = Rotation.from_quat([site_quat_local[0], site_quat_local[1], 
                                           site_quat_local[2], site_quat_local[3]])
        
        # Transform site position to world frame
        site_pos_world = body_pos + r_body.apply(site_pos_local)
        
        # Combine rotations: world = body * local
        r_world = r_body * r_site_local
        rot_matrix = r_world.as_matrix()
    else:
        # Static site in world frame
        site_pos_world = site_pos_local
        r_site = Rotation.from_quat([site_quat_local[0], site_quat_local[1], 
                                     site_quat_local[2], site_quat_local[3]])
        rot_matrix = r_site.as_matrix()
    
    return site_pos_world, rot_matrix


def get_site_position(model, state, site_idx):
    """Get site world position (convenience wrapper)."""
    pos, _ = get_site_transform(model, state, site_idx)
    return pos


def create_coordinate_axes(position, rotation_matrix, axis_length=0.05):
    """
    Create coordinate axis lines at a given position and orientation.
    
    Parameters:
        position: 3D position of the origin
        rotation_matrix: 3x3 rotation matrix
        axis_length: length of each axis line
    
    Returns:
        tuple: (starts, ends, colors) as warp arrays for viewer.log_lines()
    """
    pos = np.array(position)
    R = np.array(rotation_matrix)
    
    # Axis endpoints in world frame
    x_end = pos + axis_length * R[:, 0]  # X axis (red)
    y_end = pos + axis_length * R[:, 1]  # Y axis (green)
    z_end = pos + axis_length * R[:, 2]  # Z axis (blue)
    
    starts = wp.array([
        wp.vec3(float(pos[0]), float(pos[1]), float(pos[2])),
        wp.vec3(float(pos[0]), float(pos[1]), float(pos[2])),
        wp.vec3(float(pos[0]), float(pos[1]), float(pos[2])),
    ], dtype=wp.vec3)
    
    ends = wp.array([
        wp.vec3(float(x_end[0]), float(x_end[1]), float(x_end[2])),
        wp.vec3(float(y_end[0]), float(y_end[1]), float(y_end[2])),
        wp.vec3(float(z_end[0]), float(z_end[1]), float(z_end[2])),
    ], dtype=wp.vec3)
    
    colors = wp.array([
        wp.vec3(1.0, 0.0, 0.0),  # Red for X
        wp.vec3(0.0, 1.0, 0.0),  # Green for Y
        wp.vec3(0.0, 0.0, 1.0),  # Blue for Z
    ], dtype=wp.vec3)
    
    return starts, ends, colors


# Set initial joint positions: θ₁=0, θ₂=0, θ₃=0, θ₄=0.1
initial_q = [0.0, 0.0, 0.0, 0.1]

# Apply initial configuration
joint_q_np = state.joint_q.numpy()
for i, q_val in enumerate(initial_q):
    if i < len(joint_q_np):
        joint_q_np[i] = q_val
state.joint_q.assign(joint_q_np)

# Evaluate forward kinematics to update body positions
newton.eval_fk(model, state.joint_q, state.joint_qd, state)

print(f"Model finalized with {model.body_count} bodies and {model.joint_count} joints")
print(f"Total degrees of freedom: {model.joint_dof_count}")


### Creating the Viewer

We create a `ViewerRerun` with `keep_historical_data=True` to record the animation history. The viewer displays the robot in 3D and allows us to visualize the motion.


In [ ]:
# Create the viewer with historical data enabled
viewer = newton.viewer.ViewerRerun(keep_historical_data=True)

# Set the model (logs static geometry)
viewer.set_model(model)

# Apply correct colors from MJCF file
if shape_color_map:
    viewer.update_shape_colors(shape_color_map)
    print(f"Applied {len(shape_color_map)} shape colors from MJCF")

# Log the initial state
viewer.log_state(state)

# Add coordinate axes at ee_site
if ee_site_idx is not None:
    site_pos, site_rot = get_site_transform(model, state, ee_site_idx)
    axes_starts, axes_ends, axes_colors = create_coordinate_axes(site_pos, site_rot, axis_length=0.08)
    viewer.log_lines("/ee_site_axes", axes_starts, axes_ends, axes_colors, width=0.003)

# Display the viewer
viewer


## Part 5: Interactive FK Visualization

Let's create an interactive visualization that:
1. Generates random joint configurations
2. Computes the predicted EE position using our FK functions
3. Animates the robot smoothly to the new configuration
4. Compares the FK prediction with the actual EE position


In [ ]:
def generate_random_configuration():
    """Generate a random valid joint configuration within limits."""
    return np.random.uniform(JOINT_LIMITS[:, 0], JOINT_LIMITS[:, 1])


def interpolate_joints(q_start, q_end, alpha):
    """Linear interpolation between two joint configurations."""
    return (1 - alpha) * q_start + alpha * q_end


# Create the viewer with historical data enabled
viewer = newton.viewer.ViewerRerun(keep_historical_data=True)

# Set the model (logs static geometry)
viewer.set_model(model)

# Apply correct colors from MJCF file
if shape_color_map:
    viewer.update_shape_colors(shape_color_map)
    print(f"Applied {len(shape_color_map)} shape colors from MJCF")
    
# Animation parameters
NUM_TARGETS = 5
INTERP_STEPS = 60  # Frames per motion
FPS = 30


# Current configuration
q_current = np.array(initial_q)

sim_time = 0.0
frame_dt = 1.0 / FPS

print("Starting FK visualization...")
print(f"Generating {NUM_TARGETS} random targets")
print("=" * 60)

for target_idx in range(NUM_TARGETS):
    # Generate new random target
    q_target = generate_random_configuration()
    
    # Compute FK predictions for the target
    pos_fk_direct = fk_scara_direct(q_target)
    pos_fk_dh = fk_scara_dh(q_target)
    
    print(f"\nTarget {target_idx + 1}/{NUM_TARGETS}:")
    print(f"  Joint values: θ₁={np.degrees(q_target[0]):6.2f}°, θ₂={np.degrees(q_target[1]):6.2f}°, "
          f"θ₃={np.degrees(q_target[2]):6.2f}°, θ₄={q_target[3]*1000:6.2f}mm")
    print(f"  FK Direct: ({pos_fk_direct[0]:.4f}, {pos_fk_direct[1]:.4f}, {pos_fk_direct[2]:.4f})")
    print(f"  FK DH:     ({pos_fk_dh[0]:.4f}, {pos_fk_dh[1]:.4f}, {pos_fk_dh[2]:.4f})")
    print(f"  Difference: {np.linalg.norm(pos_fk_direct - pos_fk_dh):.2e} m")
    
    # Animate the motion
    for step in range(INTERP_STEPS):
        alpha = (step + 1) / INTERP_STEPS
        q_interp = interpolate_joints(q_current, q_target, alpha)
        
        # Update the state
        joint_q_np = state.joint_q.numpy()
        for i, q_val in enumerate(q_interp):
            if i < len(joint_q_np):
                joint_q_np[i] = q_val
        state.joint_q.assign(joint_q_np)
        
        # Evaluate FK to update body positions
        newton.eval_fk(model, state.joint_q, state.joint_qd, state)
        
        # Log to viewer
        viewer.begin_frame(sim_time)
        viewer.log_state(state)
        
        # Log coordinate axes at ee_site
        if ee_site_idx is not None:
            site_pos, site_rot = get_site_transform(model, state, ee_site_idx)
            axes_starts, axes_ends, axes_colors = create_coordinate_axes(site_pos, site_rot, axis_length=0.08)
            viewer.log_lines("/ee_site_axes", axes_starts, axes_ends, axes_colors, width=0.003)
        
        viewer.end_frame()
        
        sim_time += frame_dt
    
    # Update current position
    q_current = q_target.copy()
    
    # Brief pause at target
    for _ in range(10):
        viewer.begin_frame(sim_time)
        viewer.log_state(state)
        
        # Log coordinate axes at ee_site
        if ee_site_idx is not None:
            site_pos, site_rot = get_site_transform(model, state, ee_site_idx)
            axes_starts, axes_ends, axes_colors = create_coordinate_axes(site_pos, site_rot, axis_length=0.08)
            viewer.log_lines("/ee_site_axes", axes_starts, axes_ends, axes_colors, width=0.003)
        
        viewer.end_frame()
        sim_time += frame_dt

print("\n" + "=" * 60)
print("Animation complete!")
print("=" * 60)

viewer


## Part 5b: Comparing FK Predictions with Simulation

Let's plot the trajectory of the end-effector to visually verify that our FK calculations match the actual position from the simulation.


In [ ]:
import matplotlib.pyplot as plt
from mpl_toolkits.mplot3d import Axes3D

# Storage for trajectory comparison
trajectory_fk = []      # FK predicted positions
trajectory_sim = []     # Actual positions from simulation
joint_configs = []      # Joint configurations

# Reset to initial configuration
q_current = np.array(initial_q)
joint_q_np = state.joint_q.numpy()
for i, q_val in enumerate(q_current):
    if i < len(joint_q_np):
        joint_q_np[i] = q_val
state.joint_q.assign(joint_q_np)
newton.eval_fk(model, state.joint_q, state.joint_qd, state)

# Generate trajectory through multiple random configurations
NUM_POINTS = 50
print(f"Generating {NUM_POINTS} random configurations for comparison...")

for i in range(NUM_POINTS):
    # Generate random configuration
    q = generate_random_configuration()
    joint_configs.append(q.copy())
    
    # FK prediction
    pos_fk = fk_scara_direct(q)
    trajectory_fk.append(pos_fk.copy())
    
    # Apply to simulation and get actual position
    joint_q_np = state.joint_q.numpy()
    for j, q_val in enumerate(q):
        if j < len(joint_q_np):
            joint_q_np[j] = q_val
    state.joint_q.assign(joint_q_np)
    
    # Evaluate FK in Newton to update body positions
    newton.eval_fk(model, state.joint_q, state.joint_qd, state)
    
    # Get actual EE position from site (like MuJoCo's sensordata for ee_site)
    pos_sim = get_site_position(model, state, ee_site_idx)
    trajectory_sim.append(pos_sim.copy())

trajectory_fk = np.array(trajectory_fk)
trajectory_sim = np.array(trajectory_sim)

# Calculate errors
errors = np.linalg.norm(trajectory_fk - trajectory_sim, axis=1)
print(f"\nPosition errors between FK and simulation:")
print(f"  Mean error:    {np.mean(errors)*1000:.4f} mm")
print(f"  Max error:     {np.max(errors)*1000:.4f} mm")
print(f"  Min error:     {np.min(errors)*1000:.4f} mm")


### 4-Panel Comparison Plot

We create a 2×2 figure showing:
- **3D scatter plot**: FK predictions (blue circles) vs simulation site positions (red X marks)
- **Top view (X-Y)**: Looking down at the robot's workspace
- **Front view (X-Z)**: Side view showing vertical motion
- **Error histogram**: Distribution of position errors in millimeters


In [ ]:
# Create comparison plots
fig = plt.figure(figsize=(16, 12))

# 3D trajectory plot
ax1 = fig.add_subplot(2, 2, 1, projection='3d')
ax1.scatter(trajectory_fk[:, 0], trajectory_fk[:, 1], trajectory_fk[:, 2], 
            c='blue', s=50, alpha=0.7, label='FK Prediction', marker='o')
ax1.scatter(trajectory_sim[:, 0], trajectory_sim[:, 1], trajectory_sim[:, 2], 
            c='red', s=30, alpha=0.7, label='Simulation (Site)', marker='x')
ax1.set_xlabel('X (m)')
ax1.set_ylabel('Y (m)')
ax1.set_zlabel('Z (m)')
ax1.set_title('3D: FK Prediction vs Simulation Site Position', fontweight='bold')
ax1.legend()

# Top view (X-Y)
ax2 = fig.add_subplot(2, 2, 2)
ax2.scatter(trajectory_fk[:, 0], trajectory_fk[:, 1], 
            c='blue', s=50, alpha=0.7, label='FK Prediction', marker='o')
ax2.scatter(trajectory_sim[:, 0], trajectory_sim[:, 1], 
            c='red', s=30, alpha=0.7, label='Simulation (Site)', marker='x')
ax2.set_xlabel('X (m)')
ax2.set_ylabel('Y (m)')
ax2.set_title('Top View (X-Y): FK vs Simulation', fontweight='bold')
ax2.legend()
ax2.set_aspect('equal')
ax2.grid(True, alpha=0.3)

# Side view (X-Z)
ax3 = fig.add_subplot(2, 2, 3)
ax3.scatter(trajectory_fk[:, 0], trajectory_fk[:, 2], 
            c='blue', s=50, alpha=0.7, label='FK Prediction', marker='o')
ax3.scatter(trajectory_sim[:, 0], trajectory_sim[:, 2], 
            c='red', s=30, alpha=0.7, label='Simulation (Site)', marker='x')
ax3.set_xlabel('X (m)')
ax3.set_ylabel('Z (m)')
ax3.set_title('Front View (X-Z): FK vs Simulation', fontweight='bold')
ax3.legend()
ax3.grid(True, alpha=0.3)

# Error histogram
ax4 = fig.add_subplot(2, 2, 4)
ax4.hist(errors * 1000, bins=20, color='green', alpha=0.7, edgecolor='black')
ax4.axvline(np.mean(errors) * 1000, color='red', linestyle='--', linewidth=2, label=f'Mean: {np.mean(errors)*1000:.4f} mm')
ax4.set_xlabel('Position Error (mm)')
ax4.set_ylabel('Frequency')
ax4.set_title('Distribution of Position Errors', fontweight='bold')
ax4.legend()
ax4.grid(True, alpha=0.3)

plt.tight_layout()
plt.savefig('fk_vs_simulation_comparison.png', dpi=150, bbox_inches='tight')
plt.show()

print("\n✓ Plot saved as 'fk_vs_simulation_comparison.png'")


### Component-wise Comparison Plot

This plot shows each axis (X, Y, Z) separately, making it easy to see if there are systematic errors in any particular direction. The gray shaded area shows the error band.


In [ ]:
# Direct component-by-component comparison
fig, axes = plt.subplots(3, 1, figsize=(12, 10), sharex=True)

components = ['X', 'Y', 'Z']
colors = ['#1f77b4', '#ff7f0e', '#2ca02c']

for i, (ax, comp, color) in enumerate(zip(axes, components, colors)):
    ax.plot(trajectory_fk[:, i], 'o-', color=color, alpha=0.7, 
            label=f'FK Prediction ({comp})', markersize=6)
    ax.plot(trajectory_sim[:, i], 'x--', color='red', alpha=0.7, 
            label=f'Simulation Site ({comp})', markersize=6)
    ax.set_ylabel(f'{comp} Position (m)', fontsize=11)
    ax.legend(loc='upper right')
    ax.grid(True, alpha=0.3)
    
    # Calculate component-wise error
    comp_error = np.abs(trajectory_fk[:, i] - trajectory_sim[:, i])
    ax.fill_between(range(len(trajectory_fk)), 
                    trajectory_fk[:, i] - comp_error, 
                    trajectory_fk[:, i] + comp_error,
                    alpha=0.2, color='gray', label='Error band')

axes[0].set_title('FK Prediction vs Simulation Site Position (Component-wise)', 
                  fontsize=13, fontweight='bold')
axes[-1].set_xlabel('Sample Index', fontsize=11)

plt.tight_layout()
plt.savefig('fk_vs_simulation_components.png', dpi=150, bbox_inches='tight')
plt.show()

print("\n✓ Component comparison plot saved as 'fk_vs_simulation_components.png'")


In [ ]:
# Print detailed comparison for a few samples
print("=" * 70)
print("Detailed Comparison: FK Prediction vs Simulation Site Position")
print("=" * 70)
print(f"{'Sample':^8} | {'FK Position (m)':^30} | {'Site Position (m)':^30} | {'Error':^10}")
print("-" * 70)

# Show first 10 samples
for i in range(min(10, len(trajectory_fk))):
    fk_str = f"({trajectory_fk[i,0]:.4f}, {trajectory_fk[i,1]:.4f}, {trajectory_fk[i,2]:.4f})"
    sim_str = f"({trajectory_sim[i,0]:.4f}, {trajectory_sim[i,1]:.4f}, {trajectory_sim[i,2]:.4f})"
    err_str = f"{errors[i]*1000:.4f} mm"
    print(f"{i+1:^8} | {fk_str:^30} | {sim_str:^30} | {err_str:^10}")

print("-" * 70)
print(f"\n✓ FK predictions match simulation site positions!")
print(f"  The small errors (if any) are due to differences in the kinematic model.")


## Part 6: Full Transformation Matrix Analysis

Let's examine the complete homogeneous transformation matrix that gives us both position AND orientation of the end-effector.


In [ ]:
def full_fk_matrix(q):
    """
    Compute the complete forward kinematics transformation matrix.
    
    Returns the full 4×4 homogeneous transformation matrix T⁰_n
    that describes both position and orientation of the end-effector.
    """
    theta1, theta2, theta3, theta4 = q
    
    T01 = A_dh(a=0,  alpha=0, d=L0, theta=theta1)
    T12 = A_dh(a=L1, alpha=0, d=0, theta=theta2)
    T23 = A_dh(a=L2, alpha=0, d=0, theta=theta3)
    T34 = A_dh(a=0,  alpha=0, d=theta4, theta=0)
    
    return T01 @ T12 @ T23 @ T34


def analyze_transformation(q):
    """Analyze and display the transformation matrix components."""
    T = full_fk_matrix(q)
    
    # Extract components
    R = T[:3, :3]  # Rotation matrix
    p = T[:3, 3]   # Position vector
    
    # For SCARA, the total Z-rotation is θ₁ + θ₂ + θ₃
    total_rotation = q[0] + q[1] + q[2]
    
    print("Full Transformation Matrix T⁰_EE:")
    print(T)
    print("\nExtracted Components:")
    print(f"  Position: ({p[0]:.4f}, {p[1]:.4f}, {p[2]:.4f}) m")
    print(f"  Total Z-rotation: {np.degrees(total_rotation):.2f}°")
    print("\nRotation Matrix (3×3):")
    print(R)
    
    # Verify rotation matrix is orthonormal
    print(f"\nRotation matrix verification:")
    print(f"  det(R) = {np.linalg.det(R):.6f} (should be 1.0)")
    print(f"  R·R^T = I? Error = {np.linalg.norm(R @ R.T - np.eye(3)):.2e}")


# Analyze a sample configuration
q_sample = np.array([np.pi/6, -np.pi/8, np.pi/4, 0.05])  # 30°, -22.5°, 45°, 50mm
print("Sample configuration:")
print(f"  θ₁ = {np.degrees(q_sample[0]):.1f}°")
print(f"  θ₂ = {np.degrees(q_sample[1]):.1f}°")
print(f"  θ₃ = {np.degrees(q_sample[2]):.1f}°")
print(f"  θ₄ = {q_sample[3]*1000:.1f} mm")
print()
analyze_transformation(q_sample)


## Summary

In this tutorial, we covered:

### 1. Direct Geometric Forward Kinematics
- Simple for SCARA robots due to parallel joint axes
- Position computed from sum of link projections
- Fast to compute but less systematic

### 2. Denavit-Hartenberg Parameters
- Systematic convention for describing kinematic chains
- Four parameters per joint: a, α, d, θ
- Produces 4×4 homogeneous transformation matrices

### 3. Key DH Parameters for RRRP SCARA

| Joint | a | α | d | θ |
|-------|-----|-----|-------------|--------|
| 1 | 0 | 0 | L₀ | θ₁ |
| 2 | L₁ | 0 | 0 | θ₂ |
| 3 | L₂ | 0 | 0 | θ₃ |
| 4 | 0 | 0 | d₄ | 0 |

### 4. Verification
- Both methods produce identical results (within numerical precision)
- FK predictions match simulation results

### Next Steps
- **[Tutorial 03: Inverse Kinematics](03_inverse_kinematics.ipynb)**: Find joint values for desired EE position


### Final Test: Random Configuration

As a final demonstration, we generate one more random configuration and show the complete analysis with both FK methods agreeing on the end-effector position.


In [ ]:
# Final example: Generate and display a random configuration
print("\n" + "=" * 60)
print("FINAL EXAMPLE: Random Configuration Analysis")
print("=" * 60)

q_final = generate_random_configuration()

print(f"\nJoint Configuration:")
print(f"  θ₁ = {np.degrees(q_final[0]):7.2f}°  (Shoulder)")
print(f"  θ₂ = {np.degrees(q_final[1]):7.2f}°  (Elbow)")
print(f"  θ₃ = {np.degrees(q_final[2]):7.2f}°  (Wrist)")
print(f"  θ₄ = {q_final[3]*1000:7.2f} mm (Prismatic)")

pos_direct = fk_scara_direct(q_final)
pos_dh = fk_scara_dh(q_final)

print(f"\nEnd-Effector Position (Direct method):")
print(f"  x = {pos_direct[0]:7.4f} m")
print(f"  y = {pos_direct[1]:7.4f} m")
print(f"  z = {pos_direct[2]:7.4f} m")

print(f"\nEnd-Effector Position (DH method):")
print(f"  x = {pos_dh[0]:7.4f} m")
print(f"  y = {pos_dh[1]:7.4f} m")
print(f"  z = {pos_dh[2]:7.4f} m")

print(f"\nMethods agree: ✓ (error = {np.linalg.norm(pos_direct - pos_dh):.2e} m)")
print("=" * 60)
